# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from typing import List, Optional

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# TODO: Load environment variables
load_dotenv("/workspace/Code/project/starter/.env", override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
chroma_client = chromadb.PersistentClient(path="chromadb")

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base="https://openai.vocareum.com/v1",
    model_name="text-embedding-3-small",
)

collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

@tool
def retrieve_game(query: str) -> list:
    """Semantic search: Finds most results in the vector DB.
    args:
    - query: a question about game industry.
      Pass the user's complete question, word for word - do not shorten it to keywords.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=3)
    return [
        {
            "Platform": m.get("Platform"),
            "Name": m.get("Name"),
            "YearOfRelease": m.get("YearOfRelease"),
            "Description": m.get("Description"),
        }
        for m in results["metadatas"][0]
    ]

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
from pydantic import BaseModel

class EvaluationReport(BaseModel):
    useful: bool
    description: str

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> dict:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini")
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n"
        "Consider: do the documents answer the SPECIFIC question asked - the exact game, "
        "the exact fact, the time period? A similar game or related fact is NOT enough.\n\n"
        f"Query: {question}\n\nDocuments: {retrieved_docs}\n\n"
        'Respond ONLY with JSON, no other text: {"useful": true/false, "description": "..."}'
    )
    response = llm.invoke(prompt)
    text = response.content if hasattr(response, "content") else str(response)
    report = EvaluationReport(**json.loads(text.strip().strip("`").removeprefix("json")))
    return report.model_dump()

#### Game Web Search Tool

In [6]:
# TODO: Create game_web_search tool
from tavily import TavilyClient

@tool
def game_web_search(question: str) -> dict:
    """Web search: finds up-to-date information about video games on the web.
    Use when the internal database does not contain the answer.
    args:
    - question: a question about game industry.
    """
    try:
        client = TavilyClient(api_key=TAVILY_API_KEY)
        response = client.search(query=question, max_results=3)
        return {
            "results": [
                {"title": r["title"], "url": r["url"], "content": r["content"]}
                for r in response.get("results", [])
            ]
        }
    except Exception as e:
        return {"error": f"Web search failed: {e}"}

In [7]:
docs = retrieve_game("What platform was Pokémon Red launched on?")
print("RETRIEVED:", docs)

verdict = evaluate_retrieval("What platform was Pokémon Red launched on?", str(docs))
print("JUDGE:", verdict)

web = game_web_search("What platform was Pokémon Red launched on?")
print("WEB:", web["results"][0]["title"] if "results" in web else web)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


RETRIEVED: [{'Platform': 'Game Boy Color', 'Name': 'Pokémon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.'}, {'Platform': 'Game Boy Advance', 'Name': 'Pokémon Ruby and Sapphire', 'YearOfRelease': 2002, 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.'}, {'Platform': 'Super Nintendo Entertainment System (SNES)', 'Name': 'Super Mario World', 'YearOfRelease': 1990, 'Description': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.'}]
JUDGE: {'useful': False, 'description': 'The documents do not provide information about the platform on which Pokémon Red was launched. They only mention platforms for other Pokémon games (Gold and Silver, Ruby and Sapphire) and a completely unrelated game (Super Mario World). Therefore, they do not answer the specific query regarding Pok

### Agent

In [8]:
# TODO: Create your Agent abstraction using StateMachine
agent = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions=(
        "You are UdaPlay, an AI research assistant for the video game industry.\n"
        "For every factual question, follow this exact process:\n"
        "1. Use retrieve_game to search the internal database first.\n"
        "2. ALWAYS use evaluate_retrieval to check whether the retrieved documents "
        "actually answer the specific question - pass the question and the retrieved docs.\n"
        "3. If the evaluation says the documents are useful, answer from them.\n"
        "4. If not useful, use game_web_search to find the answer on the web.\n"
        "Always state which source your answer came from (internal database or web search), "
        "and cite it. If neither source contains the answer, say so honestly - "
        "never answer from your own knowledge."
    )
)

In [9]:
# TODO: Invoke your agent
questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for i, q in enumerate(questions, 1):
    print(f"\n{'='*60}\nQ{i}: {q}\n{'='*60}")
    run = agent.invoke(query=q, session_id=f"q{i}")
    messages = run.get_final_state()["messages"]
    for m in messages:
        role = type(m).__name__
        content = getattr(m, "content", None)
        tool_calls = getattr(m, "tool_calls", None)
        if tool_calls:
            for tc in tool_calls:
                print(f"[{role}] -> calls {tc.function.name}({tc.function.arguments})")
        elif content:
            print(f"[{role}]: {content}")


Q1: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[SystemMessage]: You are UdaPlay, an AI research assistant for the video game industry.
For every factual question, follow this exact process:
1. Use retrieve_game to search the internal database first.
2. ALWAYS use evaluate_retrieval to check whether the retrieved documents actually answer the specific question - pass the question and the retrieved docs.
3. If the evaluation says the documents are useful, answer from them.
4. If not useful, use game_web_search to find the answer on the web.
Always state which source your answer came from (internal database or web search), and cite it. If neither

In [10]:
# =====================================================
# Context retention demo: two questions, SAME session.
# The follow-up ("that game") is deliberately unanswerable
# alone - it only resolves if the agent remembers turn 1.
# =====================================================

def print_run(run):
    for m in run.get_final_state()["messages"]:
        role = type(m).__name__
        content = getattr(m, "content", None)
        tool_calls = getattr(m, "tool_calls", None)
        if tool_calls:
            for tc in tool_calls:
                print(f"[{role}] -> calls {tc.function.name}({tc.function.arguments})")
        elif content:
            print(f"[{role}]: {content}")

# --- Turn 1 ---
run_a = agent.invoke(
    query="When was Pokémon Gold and Silver released?",
    session_id="conversation_demo",
)
print("=" * 60)
print("CONTEXT DEMO - TURN 1: When was Pokémon Gold and Silver released?")
print("=" * 60)
print_run(run_a)

# --- Turn 2: same session, pronoun reference ---
run_b = agent.invoke(
    query="What platform was that game released on?",
    session_id="conversation_demo",   # SAME session_id - this is the test
)
print(f"\n{'='*60}")
print("CONTEXT DEMO - TURN 2 (same session): What platform was that game released on?")
print("=" * 60)
print_run(run_b)

print(f"\n{'='*60}")
print("CONTEXT RETENTION CHECK")
print("=" * 60)
print("Turn 2's transcript above includes the messages from Turn 1")
print("loaded into the same session, and 'that game' resolved to")
print("Pokémon Gold and Silver - demonstrating the agent retains")
print("and uses conversation context across turns.")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
CONTEXT DEMO - TURN 1: When was Pokémon Gold and Silver released?
[SystemMessage]: You are UdaPlay, an AI research assistant for the video game industry.
For every factual question, follow this exact process:
1. Use retrieve_game to search the internal database first.
2. ALWAYS use evaluate_retrieval to check whether the retrieved documents actually answer the specific question - pass the question and the retrieved docs.
3. If the evaluation says the documents are useful, answer from them.
4. If not useful, use game_web_search to find the answer on the web.
Always state which source your answer came from (internal database or web search), and c

### (Optional) Advanced

In [11]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes